# N01 — Điền NULL cột sex
**Võ Minh Huy — 2311110117**

Chỉ thực hiện quy tắc sex đã chốt. Bảng làm việc riêng; chưa xử lý lỗi số ở cột khác, chưa tạo after cuối.
## 1. Chuẩn bị
Loại 5 dòng trống và 5 bản sao sau chuẩn hóa 13 cột. Mẫu cho cố định gồm sex 0/1 đã quan sát.

In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from collections import Counter
import csv
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_csv = thu_muc / "VoMinhHuy_Heart Failure Prediction_before.csv"
with file_csv.open(encoding="utf-8-sig", newline="") as f:
    du_lieu_goc = list(csv.DictReader(f))

def doc_so(gia_tri, cot):
    chuoi = gia_tri.strip()
    if cot == "sex":
        chuoi = {"m":"1", "male":"1", "f":"0", "female":"0"}.get(chuoi.lower(), chuoi)
    try:
        return Decimal(chuoi).quantize(Decimal("0.000001"))
    except InvalidOperation:
        return None

ho_so, da_gap = [], set()
for dong in du_lieu_goc:
    if not any(v.strip() for v in dong.values()):
        continue
    ban_ghi = {cot: doc_so(v, cot) for cot, v in dong.items()}
    khoa = tuple(ban_ghi.values())
    if khoa not in da_gap:
        da_gap.add(khoa)
        ho_so.append(ban_ghi)
df = pd.DataFrame(ho_so)
df.insert(0, "MaHoSo", range(1, len(df)+1))
mau_cho = df.loc[df["sex"].isin([0, 1])].copy()
pd.DataFrame({"SoHoSo":[len(df)], "SoSexThieuTruocDien":[df["sex"].isna().sum()]})

,SoHoSo,SoSexThieuTruocDien
0,299,20


## 2. Điền sex theo mode
Thử lần lượt: (anaemia, diabetes, smoking, high_blood_pressure) → (anaemia, diabetes) → diabetes → toàn cột. Nhóm đủ 5 mẫu mới dùng; đồng mode chọn số nhỏ hơn. Không đưa ô vừa điền vào mẫu cho.

In [2]:
cac_nhom = [
    ["anaemia", "diabetes", "smoking", "high_blood_pressure"],
    ["anaemia", "diabetes"], ["diabetes"], []
]
nhat_ky = []
for i in df.index[df["sex"].isna()]:
    ket_qua = {"MaHoSo":df.at[i, "MaHoSo"],
               **{c:df.at[i,c] for c in cac_nhom[0]},
               "CapNhom":None, "SoMauCho":0, "GiaTriDien":None}
    for cap, cac_cot in enumerate(cac_nhom, start=1):
        nhom = mau_cho
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(df.at[i,cot])]
        if len(nhom) < 5:
            continue
        tan_suat = Counter(nhom["sex"])
        mode = min(v for v, n in tan_suat.items() if n == max(tan_suat.values()))
        df.at[i, "sex"] = mode
        ket_qua.update(CapNhom=cap, SoMauCho=len(nhom), GiaTriDien=mode)
        break
    nhat_ky.append(ket_qua)
nhat_ky = pd.DataFrame(nhat_ky)
tong_ket = pd.DataFrame({"SoHoSo":[len(df)],
    "SoSexThieuTruocDien":[len(nhat_ky)],
    "SoODaDien":[nhat_ky["GiaTriDien"].notna().sum()],
    "SoSexThieuSauDien":[df["sex"].isna().sum()]})
tong_ket

,SoHoSo,SoSexThieuTruocDien,SoODaDien,SoSexThieuSauDien
0,299,20,20,0


## 3. Nhật ký để chụp minh chứng
MaHoSo là số thứ tự nội bộ; SQL và Python có thể đánh số khác nhau. Đối chiếu bằng nhóm, số mẫu cho và giá trị điền.

In [3]:
nhat_ky

,MaHoSo,anaemia,diabetes,smoking,high_blood_pressure,CapNhom,SoMauCho,GiaTriDien
0,10,1.000000,0.000000,1.000000,1.000000,1,9,1.000000
1,16,1.000000,0.000000,0.000000,0.000000,1,29,1.000000
2,43,0.000000,1.000000,1.000000,1.000000,1,8,1.000000
3,65,0.000000,0.000000,0.000000,0.000000,1,32,1.000000
4,71,0.000000,0.000000,1.000000,0.000000,1,29,1.000000
5,73,0.000000,0.000000,1.000000,0.000000,1,29,1.000000
6,130,1.000000,1.000000,0.000000,0.000000,1,23,0.000000
7,132,1.000000,1.000000,0.000000,0.000000,1,23,0.000000
8,135,0.000000,0.000000,1.000000,0.000000,1,29,1.000000
9,163,1.000000,0.000000,1.000000,0.000000,1,14,1.000000


## 4. Lưu bảng kết quả của riêng sex

In [4]:
thu_muc_kq = thu_muc / "ket_qua" / "05_dien_sex"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhat_ky.to_csv(thu_muc_kq / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu thống kê riêng sex; CSV before được giữ nguyên."]})

,TrangThai
0,Đã lưu thống kê riêng sex; CSV before được giữ...
